# NB09 — Vest Detection Boost (fine-tune with new vest data)

**Problem:** Vest detection on novel real-world images is weak. The original dataset has only ~2.2% `vest` / ~2.9% `no-vest` boxes, with limited visual diversity (colors, styles, contexts).

**New data:** [Safety Vests dataset](https://universe.roboflow.com/roboflow-universe-projects/safety-vests) (Roboflow Universe, **CC BY 4.0**, mirrored on [Kaggle](https://www.kaggle.com/datasets/adilshamim8/safety-vests-detection-dataset)) — 3,897 images with `Safety Vest` / `NO-Safety Vest` boxes. Verified color variety: orange, lime, yellow, green, red vests across construction, warehouse, utility, and industrial scenes.

**Strategy:**
1. Convert CSV annotations → YOLO format, map to our class ids (`vest`=2, `no-vest`=3).
2. **Train-only merge** — new images go into training only; original val/test stay untouched so metrics remain comparable to NB05/NB07.
3. Perceptual-hash dedup — drop new images near-duplicating val/test (leakage guard, Hamming ≤ 5) or train (Hamming ≤ 2).
4. Fine-tune from the deployed YOLO11s weights (40 epochs, low LR) rather than retraining from scratch.
5. Evaluate per-class AP on the ORIGINAL test set vs the NB07 baseline, then export ONNX for deployment.

**Baseline to beat (NB07, test set):** mAP50 93.2% overall — vest 92.0%, no-vest 92.1% (in-distribution; real-world vest recall is the actual weakness).

In [ ]:
import os
from pathlib import Path

import torch

os.environ["YOLO_AUTOINSTALL"] = "false"  # stop ultralytics installing onnxruntime-gpu/numpy2 into the venv

ROOT = Path(r"c:\Users\nwagb\Desktop\SponsorshipGlobalTalentPrep\PPE_Compliance_detection")
KAGGLE_SRC = Path.home() / ".cache/kagglehub/datasets/adilshamim8/safety-vests-detection-dataset/versions/1"
VEST_IMG = ROOT / "data/vest_boost/images"
VEST_LBL = ROOT / "data/vest_boost/labels"
DATA_V2 = ROOT / "data/ppe_dataset_v2.yaml"
BASE_WEIGHTS = ROOT / "results/weights/ppe_yolo11s.pt"
CLASS_NAMES = ["hardhat", "no-hardhat", "vest", "no-vest", "person"]

print("CUDA:", torch.cuda.is_available(), "-", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")
print("base weights:", BASE_WEIGHTS.exists(), "| data v2 yaml:", DATA_V2.exists())

## 1. Data preparation

Idempotent: skips if `data/vest_boost/` is already populated. Downloads via `kagglehub` (anonymous, no API key needed for public datasets) if the Kaggle cache is missing.

In [ ]:
import csv
from collections import defaultdict

import numpy as np
from PIL import Image
import imagehash

CLASS_MAP = {"Safety Vest": 2, "NO-Safety Vest": 3}


def hash_dir(d):
    hs = []
    for f in sorted(os.listdir(d)):
        try:
            hs.append(imagehash.phash(Image.open(Path(d) / f)))
        except Exception:
            pass
    return np.array([h.hash.flatten() for h in hs], dtype=bool)


def min_dist(h, guard):
    return int(np.min(np.count_nonzero(guard != h, axis=1))) if len(guard) else 64


if any(VEST_IMG.glob("*.jpg")):
    n = len(list(VEST_IMG.glob("*.jpg")))
    print(f"vest_boost already prepared: {n} images — skipping conversion")
else:
    if not KAGGLE_SRC.exists():
        import kagglehub
        KAGGLE_SRC = Path(kagglehub.dataset_download("adilshamim8/safety-vests-detection-dataset"))
    VEST_IMG.mkdir(parents=True, exist_ok=True)
    VEST_LBL.mkdir(parents=True, exist_ok=True)

    print("hashing existing splits (leakage guard)...")
    guard_valtest = np.concatenate([
        hash_dir(ROOT / "data/processed/images/val"),
        hash_dir(ROOT / "data/processed/images/test"),
    ])
    guard_train = hash_dir(ROOT / "data/processed/images/train")

    rows_by_image = defaultdict(list)
    for split in ["train", "valid", "test"]:
        with open(KAGGLE_SRC / split / split / "_annotations.csv", newline="") as f:
            for row in csv.DictReader(f):
                rows_by_image[(split, row["filename"])].append(row)

    kept = skipped_dup = skipped_bad = 0
    box_counts = {2: 0, 3: 0}
    for (split, fname), rows in rows_by_image.items():
        src_img = KAGGLE_SRC / split / split / fname
        try:
            im = Image.open(src_img)
            im.verify()
            im = Image.open(src_img)
            w, h = im.size
            new_hash = np.array(imagehash.phash(im).hash.flatten(), dtype=bool)
        except Exception:
            skipped_bad += 1
            continue
        if min_dist(new_hash, guard_valtest) <= 5 or min_dist(new_hash, guard_train) <= 2:
            skipped_dup += 1
            continue
        lines = []
        for r in rows:
            cls = CLASS_MAP.get(r["class"])
            if cls is None:
                continue
            x1, y1, x2, y2 = (float(r[k]) for k in ("xmin", "ymin", "xmax", "ymax"))
            x1, x2 = max(0, min(x1, x2)), min(w, max(x1, x2))
            y1, y2 = max(0, min(y1, y2)), min(h, max(y1, y2))
            bw, bh = x2 - x1, y2 - y1
            if bw < 2 or bh < 2:
                continue
            lines.append(f"{cls} {(x1 + x2) / 2 / w:.6f} {(y1 + y2) / 2 / h:.6f} {bw / w:.6f} {bh / h:.6f}")
            box_counts[cls] += 1
        if not lines:
            skipped_bad += 1
            continue
        stem = Path(f"sv_{split}_{fname}").stem
        Image.open(src_img).convert("RGB").save(VEST_IMG / f"{stem}.jpg", quality=95)
        (VEST_LBL / f"{stem}.txt").write_text("\n".join(lines))
        kept += 1

    print(f"kept: {kept} | near-dup skipped: {skipped_dup} | bad/empty skipped: {skipped_bad}")
    print(f"new boxes -> vest: {box_counts[2]}, no-vest: {box_counts[3]}")

if not DATA_V2.exists():
    DATA_V2.write_text(
        f"""# v2: original dataset + vest_boost (Safety Vests, CC BY 4.0, train-only)\npath: {ROOT / 'data'}\ntrain:\n  - processed/images/train\n  - vest_boost/images\nval: processed/images/val\ntest: processed/images/test\nnc: 5\nnames:\n- hardhat\n- no-hardhat\n- vest\n- no-vest\n- person\n"""
    )
    print("wrote", DATA_V2)

In [ ]:
# Sanity check: draw boxes on a few converted samples
import random

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

samples = random.sample(sorted(VEST_IMG.glob("*.jpg")), 6)
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, img_path in zip(axes.flat, samples):
    im = Image.open(img_path)
    w, h = im.size
    ax.imshow(im)
    for line in (VEST_LBL / f"{img_path.stem}.txt").read_text().splitlines():
        c, cx, cy, bw, bh = line.split()
        c, cx, cy, bw, bh = int(c), float(cx) * w, float(cy) * h, float(bw) * w, float(bh) * h
        color = "lime" if c == 2 else "red"
        ax.add_patch(mpatches.Rectangle((cx - bw / 2, cy - bh / 2), bw, bh, fill=False, color=color, lw=2))
    ax.set_title(img_path.name[:40], fontsize=8)
    ax.axis("off")
plt.suptitle("vest_boost samples — green=vest, red=no-vest")
plt.tight_layout()
plt.show()

## 1b. Pseudo-label the missing classes (round 2)

**Round 1 failed here.** The booster dataset annotates *only* vest classes. Every
hardhat, violation and person in those 3,453 images was unlabeled, so YOLO trained
on them as **background** — teaching the model to suppress exactly the classes the
app depends on. Round 1 result: hardhat −2.6, no-vest −3.6, overall mAP50 −1.3.

The fix: let the baseline model label the classes the annotators skipped
(`hardhat`, `no-hardhat`, `person`), and merge those alongside the human vest boxes.
Human vest annotations are never overwritten; a predicted box that nearly coincides
with a human vest box (IoU > 0.75) is dropped as the same object seen twice.

**Confidence threshold = 0.45, deliberately inclusive.** Recall matters more than
precision here: a missed object stays unlabeled and trains as background, which is
the exact failure being fixed. Measured on a 60-image sample, unlabeled objects fell
from 81/34/11 (hardhat/no-hardhat/person) to **3/4/7**, and Ultralytics now reports
`0 backgrounds` across all 18,933 training images.

In [ ]:
PSEUDO_CLASSES = {0, 1, 4}   # hardhat, no-hardhat, person — the classes the booster set omits
PSEUDO_CONF = 0.45
VEST_ONLY_LBL = ROOT / "data/vest_boost/labels_vestonly"   # pristine human labels
MERGED_LBL = ROOT / "data/vest_boost/labels"               # what training reads


def _iou(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1, iy1, ix2, iy2 = max(ax1, bx1), max(ay1, by1), min(ax2, bx2), min(ay2, by2)
    inter = max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)
    if inter <= 0:
        return 0.0
    union = (ax2 - ax1) * (ay2 - ay1) + (bx2 - bx1) * (by2 - by1) - inter
    return inter / union if union > 0 else 0.0


# Idempotent: the first run moves the human vest-only labels aside, and every run
# regenerates the merged set from that pristine copy.
if not VEST_ONLY_LBL.exists():
    MERGED_LBL.rename(VEST_ONLY_LBL)
    print(f"preserved human vest labels -> {VEST_ONLY_LBL.name}/")
MERGED_LBL.mkdir(parents=True, exist_ok=True)

labeler = YOLO(BASE_WEIGHTS)
added = {0: 0, 1: 0, 4: 0}
kept_vest = suppressed = 0

for img_path in sorted(VEST_IMG.glob("*.jpg")):
    pil = Image.open(img_path)
    w, h = pil.size

    vest_lines, vest_boxes = [], []
    src = VEST_ONLY_LBL / f"{img_path.stem}.txt"
    if src.exists():
        for line in src.read_text().splitlines():
            if not line.strip():
                continue
            vest_lines.append(line)
            _, cx, cy, bw, bh = line.split()
            cx, cy, bw, bh = float(cx) * w, float(cy) * h, float(bw) * w, float(bh) * h
            vest_boxes.append((cx - bw / 2, cy - bh / 2, cx + bw / 2, cy + bh / 2))
    kept_vest += len(vest_lines)

    new_lines = []
    for b in labeler.predict(pil, imgsz=1280, conf=PSEUDO_CONF, verbose=False)[0].boxes:
        cls = int(b.cls)
        if cls not in PSEUDO_CLASSES:
            continue
        x1, y1, x2, y2 = (float(v) for v in b.xyxy[0])
        if any(_iou((x1, y1, x2, y2), vb) > 0.75 for vb in vest_boxes):
            suppressed += 1
            continue
        bw, bh = x2 - x1, y2 - y1
        if bw < 2 or bh < 2:
            continue
        new_lines.append(f"{cls} {(x1+x2)/2/w:.6f} {(y1+y2)/2/h:.6f} {bw/w:.6f} {bh/h:.6f}")
        added[cls] += 1

    (MERGED_LBL / f"{img_path.stem}.txt").write_text("\n".join(vest_lines + new_lines))

print(f"human vest boxes kept: {kept_vest}")
print(f"pseudo-labels added -> hardhat: {added[0]}, no-hardhat: {added[1]}, person: {added[4]}")
print(f"suppressed as duplicate of a vest box: {suppressed}")

## 2. Fine-tune YOLO11s from the deployed weights

20 epochs at low LR (0.002 vs 0.01 from scratch), same 1280px/batch=4 regime as NB05.
Round 1 ran 40 epochs and its validation curve was flat after ~20, so the extra half
was costing 6 hours for nothing.

Starts from the **original baseline weights**, not round 1's — round 1 baked in the
hardhat suppression, so it is not a useful starting point.

~6-7 hours on the RTX 4060 (≈20 min/epoch at 18,933 images). `torch.cuda.empty_cache()`
first: 8 GB VRAM is tight at 1280, and the run sits at 5-6.4 GB.

In [ ]:
from ultralytics import YOLO

torch.cuda.empty_cache()

model = YOLO(BASE_WEIGHTS)
results = model.train(
    data=str(DATA_V2),
    epochs=20,
    imgsz=1280,
    batch=4,
    optimizer="SGD",
    lr0=0.002,
    lrf=0.05,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=1,
    cos_lr=True,
    box=7.5,
    cls=0.5,
    dfl=1.5,
    patience=10,
    seed=42,
    device=0,
    workers=2,
    name="ppe_yolo11s_vboost_r2",
    verbose=True,
)

# Sanity check the scan output above: it must say "0 backgrounds". Any non-zero
# count means images are being trained as empty, which is what broke round 1.

## 3. Evaluate on the ORIGINAL test set

Auto-detects the newest `ppe_yolo11s_vboost*` run folder (Ultralytics suffixes re-runs). Compares per-class AP50 against the NB07 baseline.

In [ ]:
import pandas as pd

run_dirs = sorted(Path("runs/detect").glob("ppe_yolo11s_vboost_r2*"), key=os.path.getmtime)
assert run_dirs, "no round-2 run found — did training complete?"
best = run_dirs[-1] / "weights" / "best.pt"
print("evaluating:", best)

tuned = YOLO(best)
metrics = tuned.val(data=str(DATA_V2), split="test", imgsz=1280, batch=4,
                    device=0, workers=0, verbose=False)

BASELINE_AP50 = {"hardhat": 0.940, "no-hardhat": 0.966, "vest": 0.920, "no-vest": 0.921, "person": 0.915}
ROUND1_AP50 = {"hardhat": 0.914, "no-hardhat": 0.964, "vest": 0.907, "no-vest": 0.885, "person": 0.925}

new_ap = {CLASS_NAMES[int(i)]: float(ap) for i, ap in zip(metrics.box.ap_class_index, metrics.box.ap50)}
df = pd.DataFrame([
    {
        "class": c,
        "baseline": BASELINE_AP50[c],
        "round1 (broken labels)": ROUND1_AP50[c],
        "round2 (pseudo-labeled)": round(new_ap[c], 3),
        "vs baseline": round(new_ap[c] - BASELINE_AP50[c], 3),
    }
    for c in CLASS_NAMES
])
print(df.to_string(index=False))
print(f"\nmAP50: {metrics.box.map50:.3f} (baseline 0.932, round1 0.919)")
print(f"mAP50-95: {metrics.box.map:.3f} (baseline 0.639, round1 0.639)")
df.to_csv(ROOT / "results" / "nb09_r2_comparison.csv", index=False)

In [ ]:
# Real-world spot check: old model vs fine-tuned on the deployment examples
# (add your own problem images to this list — that's the real test)
spot_images = sorted((ROOT / "deployment" / "examples").glob("*.jpg"))

old = YOLO(BASE_WEIGHTS)
fig, axes = plt.subplots(len(spot_images), 2, figsize=(14, 5 * len(spot_images)))
for row, img in enumerate(spot_images):
    for col, (m, label) in enumerate([(old, "baseline"), (tuned, "vboost")]):
        r = m.predict(Image.open(img), imgsz=640, conf=0.25, verbose=False)[0]
        axes[row][col].imshow(r.plot()[..., ::-1])
        vests = sum(1 for b in r.boxes if int(b.cls) in (2, 3))
        axes[row][col].set_title(f"{img.name} — {label} ({len(r.boxes)} boxes, {vests} vest-class)", fontsize=9)
        axes[row][col].axis("off")
plt.tight_layout()
plt.show()

## 4. Export ONNX and stage for deployment

Only run once you're happy with section 3. Overwrites `deployment/ppe_yolo11s.onnx` and copies the new `.pt` to `results/weights/`. Deploy by copying the ONNX into the `PPE/` HF Space clone and pushing.

In [ ]:
import shutil

onnx_path = tuned.export(format="onnx", imgsz=640, opset=12, simplify=True)
shutil.copy(onnx_path, ROOT / "deployment" / "ppe_yolo11s.onnx")
shutil.copy(best, ROOT / "results" / "weights" / "ppe_yolo11s_vboost.pt")
print("staged deployment/ppe_yolo11s.onnx")
print("\nTo deploy:")
print("  cp deployment/ppe_yolo11s.onnx PPE/ppe_yolo11s.onnx")
print("  cd PPE && git add -A && git commit -m 'model: fine-tuned with vest_boost data' && git push")

## Notes

**The lesson from round 1 — partial labels poison a merge.** When combining a
single-purpose dataset into a multi-class one, any class the new dataset does not
annotate is silently taught as background. It is not enough for the new data to be
*correct*; it has to be *complete* for every class the model already knows. Always
check Ultralytics' scan line for `0 backgrounds`, and measure unlabeled objects by
running the current model over the new images before training.

- **Attribution (CC BY 4.0):** credit "Safety Vests dataset, Roboflow Universe" in
  the README when this ships.
- **Deploy gate:** vest and no-vest AP50 within 1 point of baseline or better,
  overall mAP50 within 1.5 points, and a clean visual spot check. Round 1 failed it
  (vest −1.3, no-vest −3.6, hardhat −2.6) and was correctly not deployed.
- If vest AP on the original test set dips slightly while real-world vest detection
  improves, that is an acceptable trade — the original test vests are narrow in
  style, and generalization is the goal. A drop in *hardhat* is not acceptable: it
  signals the background-suppression bug has returned.
- **Further variety (optional):** [SH17](https://github.com/ahmadmughees/SH17dataset)
  (8,099 Pexels images, `safety-vest` + `person`, CC BY-NC-SA — non-commercial) would
  add high-res manufacturing scenes. It has the same partial-label caveat: no
  `no-vest` class, so it needs the same pseudo-labeling treatment.
- If VRAM overflows at 1280/batch=4, drop to batch=3 — do not reduce `imgsz`, since
  small vests need the resolution.